In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")

# When executed via `nbconvert --execute`, ensure cwd is this folder
BASE_DIR = Path.cwd()
DERIVED_DIR = BASE_DIR / "derived"

PLOTS_DIR = DERIVED_DIR / "plots" / "tsageri_vs_zugdidi"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name: str, dpi: int = 200):
    out = PLOTS_DIR / f"{name}.png"
    plt.savefig(out, dpi=dpi, bbox_inches="tight")
    print("saved:", out)

TSAG_DAILY = DERIVED_DIR / "tsageri_daily_clean.csv"
TSAG_MONTHLY = DERIVED_DIR / "tsageri_monthly_clean.csv"
ZUG_DAILY = DERIVED_DIR / "zugdidi_daily_clean.csv"

OUT_HYDRO = DERIVED_DIR / "tsageri_hydro_indices.csv"
OUT_EDW_DAILY = DERIVED_DIR / "edw_daily_tsageri_vs_zugdidi.csv"

print(TSAG_DAILY)


C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\tsageri_daily_clean.csv


In [2]:
def fit_linear_trend_monthly(df: pd.DataFrame, value_col: str) -> float:
    """Return warming rate (°C/decade) using fractional-year time."""
    series = df.dropna(subset=[value_col]).copy()
    if series.empty:
        return np.nan

    years = series["year"].to_numpy() + (series["month"].to_numpy() - 0.5) / 12.0
    y = series[value_col].to_numpy()
    slope_year, _ = np.polyfit(years, y, 1)
    return float(slope_year * 10.0)


monthly = pd.read_csv(TSAG_MONTHLY)
monthly["date"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))

slope_decade = fit_linear_trend_monthly(monthly, "tmean_c_month")

plt.figure(figsize=(12, 6))
plt.plot(monthly["date"], monthly["tmean_c_month"], label="Monthly mean", alpha=0.4)
plt.plot(
    monthly["date"],
    monthly["tmean_c_month"].rolling(12, min_periods=6).mean(),
    label="12-mo mean",
    linewidth=2,
)
plt.title(f"Tsageri monthly mean T trend (warming: {slope_decade:.3f} °C/decade)")
plt.xlabel("Year")
plt.ylabel("Mean monthly temperature (°C)")
plt.legend()
plt.tight_layout()
savefig("tsageri_monthly_tmean_trend")
plt.close()


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_monthly_tmean_trend.png


In [3]:
# Seasonal cycle and heatmap
seasonal = monthly.groupby("month")["tmean_c_month"].mean().sort_index()

plt.figure(figsize=(8, 4))
plt.plot(seasonal.index, seasonal.values, marker="o")
plt.xticks(range(1, 13))
plt.title("Tsageri seasonal cycle of temperature")
plt.xlabel("Month")
plt.ylabel("Climatological mean T (°C)")
plt.tight_layout()
savefig("tsageri_seasonal_cycle")
plt.close()

pivot = monthly.pivot(index="year", columns="month", values="tmean_c_month")
plt.figure(figsize=(10, 6))
sns.heatmap(pivot, cmap="coolwarm", cbar_kws={"label": "Tmean (°C)"})
plt.title("Monthly Temperature Heatmap — Tsageri")
plt.xlabel("Month")
plt.ylabel("Year")
plt.tight_layout()
savefig("tsageri_monthly_heatmap")
plt.close()


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_seasonal_cycle.png


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_monthly_heatmap.png


In [4]:
# Freezing days per year and heatwave days per year
daily = pd.read_csv(TSAG_DAILY)
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily = daily.dropna(subset=["date"])

freezing_counts = (daily["tmean_c"] <= 0.0).groupby(daily["date"].dt.year).sum().sort_index()

plt.figure(figsize=(9, 4))
plt.plot(freezing_counts.index, freezing_counts.values, marker="o")
plt.title("Tsageri: Freezing Days per Year")
plt.xlabel("Year")
plt.ylabel("Number of freezing days")
plt.tight_layout()
savefig("tsageri_freezing_days_per_year")
plt.close()

thresh90 = float(daily["tmean_c"].quantile(0.9))
hot_counts = (daily["tmean_c"] > thresh90).groupby(daily["date"].dt.year).sum().sort_index()

plt.figure(figsize=(10, 4))
plt.bar(hot_counts.index, hot_counts.values)
plt.title(f"Tsageri: Heatwave Days per Year (threshold={thresh90:.2f} °C)")
plt.xlabel("Year")
plt.ylabel("Number of hot days")
plt.tight_layout()
savefig("tsageri_heatwave_days_per_year")
plt.close()


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_freezing_days_per_year.png


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_heatwave_days_per_year.png


In [5]:
# Hydro-climate indices (PDD, freezing days, heavy-precip days)
hydro = daily[["date", "tmean_c", "precip_mm"]].copy()
hydro["hydro_year"] = np.where(
    hydro["date"].dt.month >= 10,
    hydro["date"].dt.year + 1,
    hydro["date"].dt.year,
)

hydro["pdd_degC"] = hydro["tmean_c"].clip(lower=0.0)
pdd = (
    hydro.groupby("hydro_year")["pdd_degC"].sum().rename("pdd_degC_sum").reset_index()
)
freeze = (
    (hydro["tmean_c"] <= 0.0).groupby(hydro["hydro_year"]).sum().rename("n_freezing_days").reset_index()
)
heavy = (
    (hydro["precip_mm"] >= 20.0).groupby(hydro["hydro_year"]).sum().rename("n_days_p>=20.0mm").reset_index()
)

tsag_hydro = pdd.merge(freeze, on="hydro_year", how="outer").merge(
    heavy, on="hydro_year", how="outer"
).sort_values("hydro_year")

tsag_hydro.to_csv(OUT_HYDRO, index=False)
print("wrote:", OUT_HYDRO)

plt.figure(figsize=(10, 5))
plt.plot(tsag_hydro["hydro_year"], tsag_hydro["pdd_degC_sum"], marker="o")
plt.title("Tsageri: PDD by Hydrological Year")
plt.xlabel("Hydrological year")
plt.ylabel("PDD (°C·day)")
plt.tight_layout()
savefig("tsageri_pdd_by_hydro_year")
plt.close()

plt.figure(figsize=(10, 5))
plt.plot(tsag_hydro["hydro_year"], tsag_hydro["n_freezing_days"], marker="o", label="Freezing days")
plt.plot(
    tsag_hydro["hydro_year"],
    tsag_hydro["n_days_p>=20.0mm"],
    marker="s",
    label="Heavy precip days (>=20 mm)",
)
plt.title("Tsageri: Freezing and Heavy-Precip Days by Hydrological Year")
plt.xlabel("Hydrological year")
plt.ylabel("Days")
plt.legend()
plt.tight_layout()
savefig("tsageri_freezing_and_heavy_precip_by_hydro_year")
plt.close()


wrote: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\tsageri_hydro_indices.csv


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_pdd_by_hydro_year.png


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\tsageri_freezing_and_heavy_precip_by_hydro_year.png


In [6]:
# EDW daily comparison: Tsageri (high) vs Zugdidi 118 m (low)
zug = pd.read_csv(ZUG_DAILY)
zug["date"] = pd.to_datetime(zug["date"], errors="coerce")
zug = zug.dropna(subset=["date"])

tsag = daily[["date", "tmean_c", "elevation_m"]].rename(
    columns={"tmean_c": "tmean_hi", "elevation_m": "elev_hi_m"}
)
zug2 = zug[["date", "tmean_c", "elevation_m"]].rename(
    columns={"tmean_c": "tmean_lo", "elevation_m": "elev_lo_m"}
)

elev_hi = float(tsag["elev_hi_m"].dropna().iloc[0])
elev_lo = float(zug2["elev_lo_m"].dropna().iloc[0])
dz_100m = (elev_hi - elev_lo) / 100.0

edw = tsag.merge(zug2, on="date", how="inner")
edw["delta_t_c"] = edw["tmean_hi"] - edw["tmean_lo"]
edw["lapse_rate_c_per_100m"] = edw["delta_t_c"] / dz_100m
edw = edw.sort_values("date")

edw.to_csv(OUT_EDW_DAILY, index=False)
print("wrote:", OUT_EDW_DAILY)

edw_ts = edw.set_index("date").sort_index()
monthly_delta = edw_ts["delta_t_c"].resample("ME").mean()
monthly_lapse = edw_ts["lapse_rate_c_per_100m"].resample("ME").mean()

plt.figure(figsize=(12, 5))
plt.plot(monthly_delta.index, monthly_delta.values, alpha=0.5, label="Monthly ΔT")
plt.plot(
    monthly_delta.index,
    monthly_delta.rolling(12, min_periods=6).mean().values,
    linewidth=2,
    label="12-mo mean ΔT",
)
plt.title("EDW ΔT: Tsageri − Zugdidi (monthly means)")
plt.ylabel("ΔT (°C)")
plt.xlabel("Year")
plt.legend()
plt.tight_layout()
savefig("edw_deltaT_tsageri_minus_zugdidi")
plt.close()

plt.figure(figsize=(12, 5))
plt.plot(monthly_lapse.index, monthly_lapse.values, alpha=0.5, label="Monthly lapse rate")
plt.plot(
    monthly_lapse.index,
    monthly_lapse.rolling(12, min_periods=6).mean().values,
    linewidth=2,
    label="12-mo mean lapse rate",
)
plt.title("EDW lapse rate: Tsageri − Zugdidi (monthly means)")
plt.ylabel("Lapse rate (°C / 100 m)")
plt.xlabel("Year")
plt.legend()
plt.tight_layout()
savefig("edw_lapse_rate_tsageri_minus_zugdidi")
plt.close()


wrote: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\edw_daily_tsageri_vs_zugdidi.csv


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\edw_deltaT_tsageri_minus_zugdidi.png


saved: C:\Users\Lenovo\OneDrive\문서\Sustainability Project\Peru_Weather_Stations-20260224T230622Z-3-001\Peru_Weather_Stations\Caucasus_Weather_Stations\derived\plots\tsageri_vs_zugdidi\edw_lapse_rate_tsageri_minus_zugdidi.png
